# 13. Shor's Algorithm: Factoring via Quantum Period Finding

Shor’s algorithm achieves a polynomial-time complexity $\mathcal{O}((\log N)^3)$ for integer factorization, yielding an exponential advantage over the best-known classical algorithms, such as the General Number Field Sieve (which operates in sub-exponential time).

The algorithm does not factor the integer directly in the quantum circuit. Instead, it relies on a classical reduction of the factoring problem to the problem of **period finding** (or order finding) in modular arithmetic. The quantum processor is strictly utilized to find the period of a carefully chosen function, leveraging Quantum Phase Estimation (QPE).

This notebook explores the mathematical reduction, constructs the quantum circuit for a specific factorization ($15 = 3 \times 5$), and extracts the factors via classical post-processing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from math import gcd
from fractions import Fraction

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit.library import QFT

print("Imports Successful. Qiskit and AerSimulator are ready.")

## 13.1. Classical Reduction: Factoring to Period Finding

Let $N$ be the composite number we wish to factor. We randomly select an integer $a$ such that $1 < a < N$ and $\gcd(a, N) = 1$.

We define the periodic function:
$$f(x) = a^x \bmod N$$

By Euler's theorem, the sequence of values will eventually repeat. The **period** (or order) $r$ is the smallest non-zero integer such that:
$$a^r \equiv 1 \pmod N$$

**Why does this help factor $N$?**
If we can find $r$, and if $r$ is **even**, we can rewrite the equivalence as:
$$a^r - 1 \equiv 0 \pmod N$$
$$(a^{r/2} - 1)(a^{r/2} + 1) \equiv 0 \pmod N$$

This implies that $N$ divides the product $(a^{r/2} - 1)(a^{r/2} + 1)$. As long as $a^{r/2} \not\equiv -1 \pmod N$, neither term is a multiple of $N$ alone. Therefore, computing the greatest common divisor:
$$p = \gcd(a^{r/2} - 1, N) \quad \text{and} \quad q = \gcd(a^{r/2} + 1, N)$$
will yield non-trivial factors of $N$. Number theory guarantees that a randomly chosen $a$ satisfies these conditions with a probability $\ge \frac{1}{2}$.

In [ ]:
# Classical demonstration of the periodic function
N = 15
a = 7

# Calculate the values to observe the periodicity
x_vals = np.arange(10)
y_vals = [np.mod(a**x, N) for x in x_vals]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_vals, y_vals, linewidth=1.5, linestyle='--', marker='o', color='#042c58')
ax.set(xlabel='x', ylabel=f'{a}^x mod {N}', title='Periodicity of Modular Exponentiation')
ax.grid(True, alpha=0.3)
plt.show()

# Classically, we can see the period r = 4.
# For massive N, computing this classically is computationally intractable.

## 13.2. The Quantum Routine: Phase Estimation

Classically evaluating $f(x)$ for large $N$ requires exponential time. Shor's solution utilizes a quantum register in a superposition of all possible inputs $x$, applying the unitary operator:
$$U|y\rangle \equiv |ay \bmod N\rangle$$

By embedding this unitary within a Quantum Phase Estimation (QPE) protocol, the algorithm targets the eigenstates of $U$. The eigenvalues of $U$ take the form $e^{2\pi i (s/r)}$, where $s$ is an integer $0 \le s < r$. QPE estimates the phase $s/r$, allowing us to extract $r$.


To construct this in Qiskit for $a=7$ and $N=15$, we require an 8-qubit counting register to guarantee sufficient precision (satisfying $N^2 \le 2^t$, where $t=8$).

In [ ]:
def c_amod15(a, power):
    """
    Constructs a controlled unitary for the operation: U|y> = |a^power * y mod 15>
    Compiled explicitly for textbook demonstration.
    """
    if a not in [2, 4, 7, 8, 11, 13]:
        raise ValueError("'a' must be coprime to 15.")

    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 13]:
            U.swap(2,3); U.swap(1,2); U.swap(0,1)
        if a in [7, 8]:
            U.swap(0,1); U.swap(1,2); U.swap(2,3)
        if a in [4, 11]:
            U.swap(1,3); U.swap(0,2)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)

    U_gate = U.to_gate()
    U_gate.name = f"{a}^{power} mod 15"
    return U_gate.control()

# Initialize parameters
n_count = 8
a = 7

qc = QuantumCircuit(n_count + 4, n_count)

# 1. Initialize counting qubits to |+>
for q in range(n_count):
    qc.h(q)

# 2. Initialize target register to |1> (computational state 1)
qc.x(n_count)

# 3. Apply the controlled modular exponentiation operations
for q in range(n_count):
    qc.append(c_amod15(a, 2**q), [q] + [i + n_count for i in range(4)])

print("Modular exponentiation operations appended successfully.")

## 13.3. Quantum Fourier Transform and Measurement

The counting register now holds the sum of the phases across the superposition. To extract the phase information into the computational basis, we apply the Inverse Quantum Fourier Transform ($QFT^\dagger$) to the counting register, followed by measurement.


The measurement outcomes will be states representing binary fractions approximating $s/r$.

In [ ]:
# 4. Apply Inverse QFT to the counting register
qft_dagger = QFT(n_count, inverse=True).to_gate()
qft_dagger.name = "QFT†"
qc.append(qft_dagger, range(n_count))

# 5. Measure the counting register
qc.measure(range(n_count), range(n_count))

# Execute the simulation
simulator = AerSimulator()
transpiled_qc = transpile(qc, simulator)
result = simulator.run(transpiled_qc, shots=1024).result()
counts = result.get_counts()

# Display results
plot_histogram(counts, title="Measurement Outcomes of the Counting Register")

## 13.4. Classical Post-Processing: Continued Fractions

The observed bitstrings correspond to integers that, when divided by $2^{\text{n\_count}}$, represent the decimal phase $s/r$. We utilize the **continued fractions algorithm** to find the closest rational fraction to this decimal value. The denominator of this fraction gives us a candidate for the period $r$.

Once we identify a valid $r$ (verifying that $a^r \equiv 1 \pmod N$), we compute the factors $\gcd(a^{r/2} \pm 1, N)$.

In [ ]:
# Convert measurement outcomes to phases and extract r
phases = []
for output in counts.keys():
    decimal = int(output, 2)
    phase = decimal / (2**n_count)
    phases.append(phase)

print(f"Measured phases: {phases}")

# Use continued fractions to find r
candidates = set()
for phase in phases:
    frac = Fraction(phase).limit_denominator(15)
    r_candidate = frac.denominator
    # Verify the period
    if np.mod(a**r_candidate, N) == 1:
        candidates.add(r_candidate)

if candidates:
    r = list(candidates)[0]
    print(f"Period r found: {r}")

    # Calculate factors if r is even
    if r % 2 == 0:
        factor1 = gcd(a**(r//2) - 1, N)
        factor2 = gcd(a**(r//2) + 1, N)
        print(f"Calculated Factors: {factor1} and {factor2}")
        print(f"Validation: {factor1} * {factor2} = {factor1 * factor2}")
    else:
        print("Period is odd, algorithm failed for this choice of 'a'.")
else:
    print("Failed to find a valid period.")

## 13.5. Visualizing the algorithm



In [ ]:
from qc_interactive_education_package.quantum_library import QuantumCurriculum
from qc_interactive_education_package import launch_app

# 1. Retrieve the pre-compiled algorithms from the curriculum
algorithms = QuantumCurriculum.get_algorithms()

# 2. Extract the exact 5-qubit Grover matrix
shors_algo = algorithms["Shor's Algorithm: Factor 15 (8Q)"]

# 3. Instantiate the viewer directly into the active DOM
# The viewer will automatically deduce the initial and target states
# based on the architectural updates we implemented.
launch_app(
    mode='algorithm',
    num_qubits=8,
    preloaded_circuit=shors_algo,
    show_circuit=True,
    show_annotations=True,
    show_final_state = False
)

Initializing Quantum Education Suite SPA (Algorithm Mode)...
Starting local server... A browser window will open automatically once ready.


## 13.6. Critical Evaluation and Alternatives

While Shor's algorithm provides a mathematically sound proof of quantum advantage, its practical execution carries profound caveats that necessitate realistic appraisal:

1. **Hardware Constraints:** Factoring cryptographic keys (e.g., RSA-2048) requires millions of physical qubits to encode thousands of error-corrected logical qubits. Current Noisy Intermediate-Scale Quantum (NISQ) devices lack the coherence times and gate fidelities necessary to execute deep QPE routines.
2. **Modular Exponentiation Overhead:** The bottleneck of the algorithm is compiling the $U^{2^q}$ gates. Constructing a generic, efficient quantum arithmetic circuit for large arbitrary $N$ involves immense spatial and temporal complexity.

**Cryptographic Alternatives:**
The eventual viability of Shor's algorithm threatens asymmetric systems reliant on integer factorization and discrete logarithms (RSA, ECC). The rigorous solution is a migration to **Post-Quantum Cryptography (PQC)**. Cryptosystems built on lattice problems (e.g., Learning With Errors), hash-based signatures, and multivariate quadratic equations represent alternative protocols that currently remain intractable for both classical architectures and Shor's formulation.